# Assignment 4: Types of Supervised Machine Learning

In this assignment, I will train and compare three different models for each type of supervised learning:
- Classification (Wine dataset): Logistic Regression, Support Vector Classifier (SVC), and K-Nearest Neighbors Classifier
- Regression (Car Prices dataset): Linear Regression, Support Vector Regressor (SVR), and K-Nearest Neighbors Regressor

For each dataset, I will load and explore the data, split it into training and test sets, train the three models, and compare their performance. Then I will tune the hyperparameters of each model and report how the changes affect performance.

Note: The Wine dataset classes are three different grape cultivars, not quality levels. For the regression models, I use R² and RMSE (Root Mean Squared Error) instead of accuracy, because accuracy only applies to classification.

# Part A: Classification with the Wine Dataset

## Step 1: Load and Explore the Dataset

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Regression models (for Part B)
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
pd.set_option("display.max_columns", None)


In [3]:
# Load the Wine dataset
wine = load_wine(as_frame=True)
wine_df = wine.frame

In [4]:
print("Shape:", wine_df.shape)

Shape: (178, 14)


In [7]:
print("\nClass counts:")
print(wine_df["target"].value_counts().sort_index())



Class counts:
target
0    59
1    71
2    48
Name: count, dtype: int64


In [8]:
print("\nMissing values:", wine_df.isnull().sum().sum())
wine_df.describe().round(2)


Missing values: 0


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
count,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00
mean,13.00,2.34,2.37,19.49,99.74,2.30,2.03,0.36,1.59,5.06,0.96,2.61,746.89,0.94
std,0.81,1.12,0.27,3.34,14.28,0.63,1.00,0.12,0.57,2.32,0.23,0.71,314.91,0.78
min,11.03,0.74,1.36,10.60,70.00,0.98,0.34,0.13,0.41,1.28,0.48,1.27,278.00,0.00
25%,12.36,1.60,2.21,17.20,88.00,1.74,1.20,0.27,1.25,3.22,0.78,1.94,500.50,0.00
50%,13.05,1.87,2.36,19.50,98.00,2.36,2.13,0.34,1.56,4.69,0.96,2.78,673.50,1.00
75%,13.68,3.08,2.56,21.50,107.00,2.80,2.88,0.44,1.95,6.20,1.12,3.17,985.00,2.00
max,14.83,5.80,3.23,30.00,162.00,3.88,5.08,0.66,3.58,13.00,1.71,4.00,1680.00,2.00


### Interpretation

I imported all the libraries for both parts of the assignment, including the three classification models (Logistic Regression, SVC, and KNN Classifier), the three regression models (Linear Regression, SVR, and KNN Regressor), GridSearchCV for hyperparameter tuning, and the evaluation metrics.

The Wine dataset has 178 rows and 14 columns: 13 chemical features and 1 target column. There are no missing values.

The three classes are:
- Class 0: 59 wines
- Class 1: 71 wines
- Class 2: 48 wines

The classes are fairly balanced, so I do not need techniques like SMOTE. From the summary statistics, the features are on very different scales (for example, proline is in the hundreds or thousands, while hue is below 2). This matters for SVC and KNN, which are sensitive to feature scale, so I will scale the features before training.

## Step 2: Split Data into Training and Test Sets

In [9]:
X_wine = wine_df.drop("target", axis=1)
y_wine = wine_df["target"]

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

print("Training set:", X_train_w.shape)
print("Test set:", X_test_w.shape)

Training set: (142, 13)
Test set: (36, 13)


## Steps 3 to 5: Train Multiple Models and Evaluate Their Performance

In [11]:
# Each model is a Pipeline: scale first, then the model
clf_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "SVC": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(random_state=42))
    ]),
    "KNN Classifier": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ])
}

clf_results = []

for name, model in clf_models.items():
    model.fit(X_train_w, y_train_w)                 
    y_pred = model.predict(X_test_w)              
    train_acc = accuracy_score(y_train_w, model.predict(X_train_w))
    test_acc = accuracy_score(y_test_w, y_pred)
    clf_results.append({"Model": name,
                        "Train Accuracy": round(train_acc, 4),
                        "Test Accuracy": round(test_acc, 4)})
    print(f"===== {name} =====")
    print(classification_report(y_test_w, y_pred, target_names=wine.target_names))

clf_results_df = pd.DataFrame(clf_results)
print(clf_results_df)

===== Logistic Regression =====
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36

===== SVC =====
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36

===== KNN Classifier =====
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       1.00      0.93      0.96        14


### Interpretation

I trained three classification models using their default settings. Each model was placed in a Pipeline with StandardScaler, so the features were scaled using only the training data.

Results:
- Logistic Regression: training accuracy 100%, test accuracy 97.22%
- SVC: training accuracy 99.30%, test accuracy 97.22%
- KNN Classifier (k = 5): training accuracy 97.89%, test accuracy 97.22%

All three models reached the same test accuracy of 97.22%, which means each one classified 35 out of 36 test wines correctly and made only one mistake. The mistakes were different:
- Logistic Regression and SVC both misclassified one class_2 wine as class_1. This is why their class_2 recall is 0.90 and class_1 precision is 0.93.
- KNN misclassified one class_1 wine as class_2. This is why its class_1 recall is 0.93 and class_2 precision is 0.91.

Which model performs best?
On the test set, the three models are tied, so I cannot pick a clear winner from these results alone. The test set has only 36 wines, so a single prediction changes accuracy by about 2.8%.

Looking at training accuracy, Logistic Regression scored 100% on the training data, which could be a small sign of overfitting, while KNN had the smallest gap between training and test accuracy. All three models perform well because the Wine dataset has clear chemical differences between the three cultivars, and scaling the features helped SVC and KNN in particular, since they depend on distances.

To compare the models more fairly, I will use cross-validation and hyperparameter tuning in the next step.

## Step 6: Parameter Tuning and Optimization

In [12]:
# Hyperparameters to try for each model
clf_param_grids = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100]
    },
    "SVC": {
        "model__C": [0.1, 1, 10, 100],
        "model__kernel": ["linear", "rbf"]
    },
    "KNN Classifier": {
        "model__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21],
        "model__weights": ["uniform", "distance"]
    }
}

clf_tuned_results = []

for name, model in clf_models.items():
    grid = GridSearchCV(model, clf_param_grids[name], cv=5, scoring="accuracy")
    grid.fit(X_train_w, y_train_w)

    test_acc = accuracy_score(y_test_w, grid.predict(X_test_w))
    clf_tuned_results.append({"Model": name,
                              "Best Parameters": grid.best_params_,
                              "Best CV Accuracy": round(grid.best_score_, 4),
                              "Test Accuracy": round(test_acc, 4)})

    # Show how each parameter setting performed
    print(f"===== {name} =====")
    cv_table = pd.DataFrame(grid.cv_results_)[["params", "mean_test_score"]]
    cv_table["mean_test_score"] = cv_table["mean_test_score"].round(4)
    print(cv_table.to_string(index=False))
    print()

clf_tuned_df = pd.DataFrame(clf_tuned_results)
print(clf_tuned_df.to_string(index=False))

===== Logistic Regression =====
            params  mean_test_score
{'model__C': 0.01}           0.9791
 {'model__C': 0.1}           0.9931
   {'model__C': 1}           0.9931
  {'model__C': 10}           0.9860
 {'model__C': 100}           0.9860

===== SVC =====
                                      params  mean_test_score
{'model__C': 0.1, 'model__kernel': 'linear'}           0.9862
   {'model__C': 0.1, 'model__kernel': 'rbf'}           0.9793
  {'model__C': 1, 'model__kernel': 'linear'}           0.9862
     {'model__C': 1, 'model__kernel': 'rbf'}           0.9862
 {'model__C': 10, 'model__kernel': 'linear'}           0.9862
    {'model__C': 10, 'model__kernel': 'rbf'}           0.9862
{'model__C': 100, 'model__kernel': 'linear'}           0.9862
   {'model__C': 100, 'model__kernel': 'rbf'}           0.9862

===== KNN Classifier =====
                                                  params  mean_test_score
  {'model__n_neighbors': 1, 'model__weights': 'uniform'}           0.9367
 